# MLDL2 Homework 3

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.models as models
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from tqdm.auto import tqdm
import os
import numpy as np
import torchvision.transforms as transforms
import torch.optim as optim

# Device 설정
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

In [ ]:
# Hyperparameters
batch_size = 64
learning_rate = 0.0002
num_epochs = 7

# Dataset class
class CUB_Dataset(Dataset):
    def __init__(self, img_file, label_file, transform=None):
        self.img = np.load(img_file)
        self.labels = np.load(label_file)
        self.transform = transform

    def __len__(self):
        return len(self.img)

    def __getitem__(self, idx):
        image = self.img[idx]
        if self.transform:
            image = self.transform(image)
        label = self.labels[idx]
        return image, label

# Data transforms
cub_bird_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))
])

In [ ]:
# Load datasets
cub_train_dataset = CUB_Dataset(img_file="/content/drive/MyDrive/AI인턴십/CUB_train_images.npy",
                                label_file="/content/drive/MyDrive/AI인턴십/CUB_train_labels.npy", transform=cub_bird_transform)
cub_train_loader = torch.utils.data.DataLoader(cub_train_dataset, batch_size=batch_size, shuffle=True)

cub_val_dataset = CUB_Dataset(img_file="/content/drive/MyDrive/AI인턴십/CUB_val_images.npy",
                              label_file="/content/drive/MyDrive/AI인턴십/CUB_val_labels.npy", transform=cub_bird_transform)
cub_val_loader = torch.utils.data.DataLoader(cub_val_dataset, batch_size=batch_size, shuffle=False)


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.optim.lr_scheduler as lr_scheduler
from torchvision import models
from tqdm import tqdm
import os

# Hyperparameters
learning_rate = 0.0002
num_epochs = 7

# Device configuration
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

# ResNet 모델 정의
model = models.resnet50(pretrained=False)
model.fc = nn.Linear(model.fc.in_features, 200)  # 200개의 클래스를 위한 최종 레이어
model = model.to(device)

# Pre-training 모델 로드
pretrained_model_path = "/content/drive/MyDrive/AI인턴십/model/pretrained_resnet_cifar_best_10.pth"
checkpoint = torch.load(pretrained_model_path)
model.load_state_dict(checkpoint['model_state_dict'])

# 손실 함수 정의
criterion = nn.CrossEntropyLoss()

# 옵티마이저 정의
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

# 학습률 스케줄러 정의
scheduler = lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.1, patience=5, verbose=True)

# 최적의 손실 값을 무한대로 초기화
best_loss = float('inf')

# 체크포인트를 저장할 디렉토리 경로
checkpoint_dir = "/content/drive/MyDrive/AI인턴십/model/"

# Fine-tuning loop
for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for inputs, labels in tqdm(cub_train_loader):
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    epoch_loss = running_loss / len(cub_train_loader)
    epoch_acc = 100 * correct / total

    print(f"Epoch [{epoch + 1}/{num_epochs}], Loss: {epoch_loss:.4f}, Acc: {epoch_acc:.2f}%")

    # 검증 손실에 따라 학습률 감소
    scheduler.step(epoch_loss)

    # 최적 모델 저장
    if epoch_loss < best_loss:
        best_loss = epoch_loss
        best_epoch = epoch + 1
        best_model_path = os.path.join(checkpoint_dir, f"trained_resnet_cub_ffffinal_{best_epoch}.pth")
        torch.save({
            'epoch': epoch + 1,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'loss': best_loss,
        }, best_model_path)
        print(f"New best model saved at epoch {best_epoch} with loss: {best_loss:.4f}")

print('Finished Fine-tuning')


/usr/local/lib/python3.10/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)
<ipython-input-8-3e07a187c08e>:23: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions

Epoch [1/7], Loss: 5.7526, Acc: 1.40%
New best model saved at epoch 1 with loss: 5.7526


100%|██████████| 94/94 [00:59<00:00,  1.58it/s]


Epoch [2/7], Loss: 4.9330, Acc: 4.47%
New best model saved at epoch 2 with loss: 4.9330


100%|██████████| 94/94 [01:00<00:00,  1.55it/s]


...

Epoch [5/7], Loss: 2.0840, Acc: 50.02%
New best model saved at epoch 5 with loss: 2.0840


100%|██████████| 94/94 [01:01<00:00,  1.54it/s]


Epoch [6/7], Loss: 1.4707, Acc: 62.25%
New best model saved at epoch 6 with loss: 1.4707


100%|██████████| 94/94 [01:02<00:00,  1.51it/s]


Epoch [7/7], Loss: 0.9919, Acc: 76.13%
New best model saved at epoch 7 with loss: 0.9919
Finished Fine-tuning


In [ ]:
aa

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models
from torch.utils.data import DataLoader

# ResNet 모델 정의
model = models.resnet50(pretrained=False)
model.fc = nn.Linear(model.fc.in_features, 200)  # 200개의 클래스를 위한 최종 레이어
model = model.to(device)

# 저장된 pre-training 불러오기
checkpoint_path = "/content/drive/MyDrive/AI인턴십/trained_resnet_cub_final_9.pth"
checkpoint = torch.load(checkpoint_path)
model.load_state_dict(checkpoint['model_state_dict'])

# 모델 검증 (예시 코드)
model.eval()
correct = 0
total = 0
with torch.no_grad():
    for data in cub_val_loader:
        images, labels = data
        outputs = model(images.to(device))
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted.cpu() == labels).sum().item()

print(f'Accuracy of the network on the 2897 validation images: {100 * correct / total:.2f} %')


<ipython-input-31-397736adcf1e>:14: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(checkpoint_path)


Accuracy of the network on the 2897 validation images: 8.73 %


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models
from torch.utils.data import DataLoader

# ResNet 모델 정의
model = models.resnet50(pretrained=False)
model.fc = nn.Linear(model.fc.in_features, 200)  # 200개의 클래스를 위한 최종 레이어
model = model.to(device)

# 저장된 pre-training 불러오기
checkpoint_path = "/content/drive/MyDrive/AI인턴십/trained_resnet_cub_final_8.pth"
checkpoint = torch.load(checkpoint_path)
model.load_state_dict(checkpoint['model_state_dict'])

# 모델 검증 (예시 코드)
model.eval()
correct = 0
total = 0
with torch.no_grad():
    for data in cub_val_loader:
        images, labels = data
        outputs = model(images.to(device))
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted.cpu() == labels).sum().item()

print(f'Accuracy of the network on the 2897 validation images: {100 * correct / total:.2f} %')


<ipython-input-34-f324fea5471c>:14: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(checkpoint_path)


Accuracy of the network on the 2897 validation images: 8.49 %


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models
from torch.utils.data import DataLoader

# ResNet 모델 정의
model = models.resnet50(pretrained=False)
model.fc = nn.Linear(model.fc.in_features, 200)  # 200개의 클래스를 위한 최종 레이어
model = model.to(device)

# 저장된 pre-training 불러오기
checkpoint_path = "/content/drive/MyDrive/AI인턴십/trained_resnet_cub_final_11.pth"
checkpoint = torch.load(checkpoint_path)
model.load_state_dict(checkpoint['model_state_dict'])

# 모델 검증 (예시 코드)
model.eval()
correct = 0
total = 0
with torch.no_grad():
    for data in cub_val_loader:
        images, labels = data
        outputs = model(images.to(device))
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted.cpu() == labels).sum().item()

print(f'Accuracy of the network on the 2897 validation images: {100 * correct / total:.2f} %')


<ipython-input-35-5f41d4676e65>:14: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(checkpoint_path)


Accuracy of the network on the 2897 validation images: 8.87 %


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models
from torch.utils.data import DataLoader

# ResNet 모델 정의
model = models.resnet50(pretrained=False)
model.fc = nn.Linear(model.fc.in_features, 200)  # 200개의 클래스를 위한 최종 레이어
model = model.to(device)

# 저장된 pre-training 불러오기
checkpoint_path = "/content/drive/MyDrive/AI인턴십/trained_resnet_cub_final_6.pth"
checkpoint = torch.load(checkpoint_path)
model.load_state_dict(checkpoint['model_state_dict'])

# 모델 검증 (예시 코드)
model.eval()
correct = 0
total = 0
with torch.no_grad():
    for data in cub_val_loader:
        images, labels = data
        outputs = model(images.to(device))
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted.cpu() == labels).sum().item()

print(f'Accuracy of the network on the 2897 validation images: {100 * correct / total:.2f} %')


<ipython-input-41-9031c6677603>:14: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(checkpoint_path)


Accuracy of the network on the 2897 validation images: 9.29 %


# 6. Test and Submit  

You can modify your TestDataset, but you should be mindful to align it with the training dataset and its transformations.

In [ ]:
# 테스트 데이터셋 로드
BATCH_SIZE = 64

class TestDataset(Dataset):
    def __init__(self, img_file, transform=None):
        self.img = np.load(img_file)
        self.transform = transform

    def __len__(self):
        return len(self.img)

    def __getitem__(self, idx):
        image = self.img[idx]
        if self.transform is not None:
            image = self.transform(image)
        return image

test_transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))
])

test_dataset = TestDataset(img_file="/content/drive/MyDrive/AI인턴십/CUB_test_images.npy", transform=test_transform)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)


## **Do not modify the cell below!!!!**


In [ ]:
def test(model, test_loader):
  model.eval()
  test_predictions = []

  with torch.inference_mode():
      for i, data in enumerate(tqdm(test_loader)):
          data = data.float().to(device)
          output = model(data)
          test_predictions.append(output.cpu())

  return torch.cat(test_predictions, dim=0)

In [ ]:
# Save test output npy file
predictions = test(model, test_loader)
np.save('/content/drive/MyDrive/AI인턴십/model/3조_Final_RESULT', predictions.numpy())

100%|██████████| 46/46 [00:09<00:00,  4.78it/s]
